In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, brier_score_loss


In [3]:
heart = pd.read_csv("heart_disease_uci.csv")

Let's look into the data and do some modifications, and also try to think about which would be good candidate features for the models

In [4]:
heart.head()

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


In [5]:
# column num values:
heart["num"].unique()

array([0, 2, 1, 3, 4], dtype=int64)

On this dataset, the column num has values in $\{0,1,2,3,4\}$, where $0 =$ no heart disease and $1,2,3,4$ indicate the person having heart disease $\\$ with $1 =$ mild, $2 =$ moderate, $3 =$ severe, $4 =$ very severe heart disease. $\\$ We turn this into classification $0,1$ problem where we try to classificate $0=$ no heart disease, $1=$ has heart disease, that is, we make a new column disease as such: $\\$  if column num $=0$ then disease $=0$ and if num $\in \{1,2,3,4\}$ then disease $=1$.


In [6]:
# make a new column "disease" where disease = 0 when num = 0 and disease = 1 when num = 1,2,3 or 4.
heart["disease"] = 0
heart.loc[heart["num"]>0, "disease"] = 1
pd.crosstab(heart["disease"], heart["num"])
heart.loc[heart["chol"]==0,"chol"]=np.nan
heart.loc[heart["trestbps"]==0,"trestbps"]=np.nan



In [7]:
#create train_test_split for future models
X = heart.drop(columns=["id","disease","num","dataset"])
y = heart["disease"]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=10)

Let's do some pre inspection on the data to familiarize ourselves with it

In [8]:
# male/female distribution
heart["sex"].value_counts()
pd.crosstab(heart["disease"],heart["sex"])

sex,Female,Male
disease,,
0,144,267
1,50,459


In [9]:
#create age_group column to split age into groups
heart["age_group"] = pd.cut(heart["age"],bins=range(0,101,10),right=False)
heart["age_group"].value_counts()

tab = pd.crosstab(heart["age_group"],heart["disease"], normalize="index").mul(100).round(1)
tab


disease,0,1
age_group,,
"[20, 30)",100.0,0.0
"[30, 40)",65.8,34.2
"[40, 50)",59.9,40.1
"[50, 60)",43.2,56.8
"[60, 70)",26.6,73.4
"[70, 80)",29.0,71.0


It very much seems like age is a clear factor in heart diseases: the higher the age, the bigger chance to have a heart disease.

In [10]:
pd.crosstab(heart["cp"],heart["disease"])

disease,0,1
cp,,
asymptomatic,104,392
atypical angina,150,24
non-anginal,131,73
typical angina,26,20


An interesting finding is that the asymptomatic group has the highest proportion of individuals with heart disease: approximately 79% of individuals in this category have the disease. This suggests that the absence of reported chest pain does not necessarily indicate the absence of heart disease. Conversely, many individuals in the other chest-pain categories do not have the disease. One possible explanation is that chest pain can have non-cardiac causes, such as gastroesophageal reflux disease (GERD) or musculoskeletal conditions. However, these associations should be interpreted cautiously, as the table alone cannot establish the underlying causes or explain why the observed pattern occurs.

In [11]:
heart["chol"].describe()
heart["chol"].value_counts()

chol
220.0    10
254.0    10
219.0     9
216.0     9
204.0     9
         ..
412.0     1
358.0     1
360.0     1
321.0     1
385.0     1
Name: count, Length: 216, dtype: int64

Chol has a lot of missing values, requires inputation

chol, age, cp, sex, exang

In [12]:
#crosstable of excercise induced chest pain and disease
pd.crosstab(heart["exang"],heart["disease"], normalize="index").mul(100).round(1)


disease,0,1
exang,,
False,63.6,36.4
True,16.3,83.7


Exang = exercise-induced angina (True/ False), seems to have positive correlation with disease

In [13]:
#correlation between excercise induced chest pain and disease
heart[["exang","disease"]].corr()
heart["exang"].mode()

0    False
Name: exang, dtype: object

Exang seems to be promising feature for model.

In [14]:
X.describe(include="all")

,age,sex,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal
count,920.000000,920,920,860.000000,718.000000,830,918,865.000000,865,858.000000,611,309.000000,434
unique,NaN,2,4,NaN,NaN,2,3,NaN,2,NaN,3,NaN,3
top,NaN,Male,asymptomatic,NaN,NaN,False,normal,NaN,False,NaN,flat,NaN,normal
freq,NaN,726,496,NaN,NaN,692,551,NaN,528,NaN,345,NaN,196
mean,53.510870,NaN,NaN,132.286047,246.832869,NaN,NaN,137.545665,NaN,0.878788,NaN,0.676375,NaN
std,9.424685,NaN,NaN,18.536175,58.527062,NaN,NaN,25.926276,NaN,1.091226,NaN,0.935653,NaN
min,28.000000,NaN,NaN,80.000000,85.000000,NaN,NaN,60.000000,NaN,-2.600000,NaN,0.000000,NaN
25%,47.000000,NaN,NaN,120.000000,210.000000,NaN,NaN,120.000000,NaN,0.000000,NaN,0.000000,NaN
50%,54.000000,NaN,NaN,130.000000,239.500000,NaN,NaN,140.000000,NaN,0.500000,NaN,0.000000,NaN
75%,60.000000,NaN,NaN,140.000000,276.750000,NaN,NaN,157.000000,NaN,1.500000,NaN,1.000000,NaN


In [15]:
categorical = ["sex","cp","fbs","restecg","exang","slope","ca","thal"]
numerical=[]
for i in X.columns:
    if i not in categorical:
        numerical.append(i)






Make a preprocessor for data columns which handles imputation

In [16]:
preprocessor = make_column_transformer(
    (Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())]),numerical),
(Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("encoder",OneHotEncoder(handle_unknown="ignore"))]),categorical)
)


Make a model now and cross-validate

In [17]:
model = Pipeline([("preprocessor",preprocessor), ("classifier",LogisticRegression(max_iter=1000))])

cv = StratifiedKFold(n_splits=10,shuffle=True,random_state=10)
scores = cross_validate(model,X_train,y_train,cv=cv, scoring=["accuracy", "precision", "recall", "roc_auc","neg_brier_score"])

for metric in ["accuracy", "precision", "recall", "roc_auc","neg_brier_score"]:
    values = scores[f"test_{metric}"]
    if metric== "neg_brier_score":
        values = -values
        metric = "brier_score"
    print(f"{metric}: {values.mean():.3f} +- {values.std():.3f}")

accuracy: 0.821 +- 0.031
precision: 0.835 +- 0.041
recall: 0.845 +- 0.029
roc_auc: 0.892 +- 0.025
brier_score: 0.129 +- 0.015


A baseline model that has all possible variables as features, has the following metrics after 10-fold Cross-Validation. In a classification like this, recall is very important. To clarify what recall is: Recall captures how many actual diseases the model found. It can be calculated via formula $\frac{TP}{TP+FN}$, where $TP = \text{True positive}$ and $FN = \text{False Negative}$.

accuracy: 0.821 +- 0.031
precision: 0.835 +- 0.041
recall: 0.845 +- 0.029
roc_auc: 0.892 +- 0.025
brier_score: 0.129 +- 0.015

Next, try with different features. Select: age,sex,cp,trestbps,thalch,exang,oldpeak,ca

In [18]:
categorical = ["sex","cp","exang","ca"]
numerical = ["age","trestbps","thalch","oldpeak"]
features = categorical+numerical

preprocessor = make_column_transformer(
    (Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())]),numerical),
(Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("encoder",OneHotEncoder(handle_unknown="ignore"))]),categorical)
)

model = Pipeline([("preprocessor",preprocessor), ("classifier",LogisticRegression(max_iter=1000))])

cv = StratifiedKFold(n_splits=10,shuffle=True,random_state=10)
scores = cross_validate(model,X_train[features],y_train,cv=cv, scoring=["accuracy", "precision", "recall", "roc_auc","neg_brier_score"])

for metric in ["accuracy", "precision", "recall", "roc_auc","neg_brier_score"]:
    values = scores[f"test_{metric}"]
    if metric== "neg_brier_score":
        values = -values
        metric = "brier_score"
    print(f"{metric}: {values.mean():.3f} +- {values.std():.3f}")

accuracy: 0.812 +- 0.030
precision: 0.829 +- 0.038
recall: 0.835 +- 0.035
roc_auc: 0.883 +- 0.028
brier_score: 0.135 +- 0.017


accuracy: 0.812 +- 0.030
precision: 0.829 +- 0.038
recall: 0.835 +- 0.035
roc_auc: 0.883 +- 0.028
brier_score: 0.135 +- 0.017

Very similar to the model with all features, one would prefer this over the full model for simplicity. Let's experiment more by adding "thal" and "slope", so the next test has 10 features

In [19]:
categorical = ["sex","cp","exang","ca","thal","slope"]
numerical = ["age","trestbps","thalch","oldpeak"]
features = categorical+numerical

preprocessor = make_column_transformer(
    (Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())]),numerical),
(Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("encoder",OneHotEncoder(handle_unknown="ignore"))]),categorical)
)

model = Pipeline([("preprocessor",preprocessor), ("classifier",LogisticRegression(max_iter=1000))])

cv = StratifiedKFold(n_splits=10,shuffle=True,random_state=10)
scores = cross_validate(model,X_train[features],y_train,cv=cv, scoring=["accuracy", "precision", "recall", "roc_auc","neg_brier_score"])

for metric in ["accuracy", "precision", "recall", "roc_auc","neg_brier_score"]:
    values = scores[f"test_{metric}"]
    if metric== "neg_brier_score":
        values = -values
        metric = "brier_score"
    print(f"{metric}: {values.mean():.3f} +- {values.std():.3f}")
    



accuracy: 0.812 +- 0.029
precision: 0.824 +- 0.039
recall: 0.843 +- 0.029
roc_auc: 0.887 +- 0.027
brier_score: 0.133 +- 0.017


accuracy: 0.812 +- 0.029
precision: 0.824 +- 0.039
recall: 0.843 +- 0.029
roc_auc: 0.887 +- 0.027
brier_score: 0.133 +- 0.017

All of the models are pretty similar, in all honestly you could choose any, I would choose the one with all features or the 10-feature model. Next, let's try random forest with all features.

In [20]:
from sklearn.ensemble import RandomForestClassifier

categorical = ["sex","cp","fbs","restecg","exang","slope","ca","thal"]
numerical = [column for column in X_train.columns if column not in categorical]

preprocessor_trees= make_column_transformer(
    (Pipeline([("imputer",SimpleImputer(strategy="median"))]),numerical),
(Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("encoder",OneHotEncoder(handle_unknown="ignore"))]),categorical)
)

rfmodel = Pipeline([("preprocessor",preprocessor_trees), ("classifier",RandomForestClassifier(n_estimators=100,max_depth=3,random_state=10))])

cv = StratifiedKFold(n_splits=10,shuffle=True,random_state=10)
scores = cross_validate(rfmodel,X_train,y_train,cv=cv, scoring=["accuracy", "precision", "recall", "roc_auc","neg_brier_score"])

for metric in ["accuracy", "precision", "recall", "roc_auc","neg_brier_score"]:
    values = scores[f"test_{metric}"]
    if metric== "neg_brier_score":
        values = -values
        metric = "brier_score"
    print(f"{metric}: {values.mean():.3f} +- {values.std():.3f}")



accuracy: 0.799 +- 0.038
precision: 0.800 +- 0.036
recall: 0.850 +- 0.050
roc_auc: 0.886 +- 0.034
brier_score: 0.145 +- 0.014


Let's loop over different values for n_estimators and max_depth to find best recall. I value recall in this task because not missing heart conditions is critical.

In [21]:
from sklearn.model_selection import ParameterGrid
param_grid = {"classifier__n_estimators": [100,200,500],
              "classifier__max_depth":[3,5,8,None]}
results=[]
for params in ParameterGrid(param_grid):
    rfmodel.set_params(**params)
    scores = cross_validate(rfmodel,X_train,y_train,cv=cv, scoring=["accuracy", "precision", "recall", "roc_auc","neg_brier_score"])
    results.append({"n_estimators": params["classifier__n_estimators"],
                    "max_depth": params["classifier__max_depth"],
                    "accuracy": scores["test_accuracy"].mean(),
                    "precision":scores["test_precision"].mean(),
                    "recall": scores["test_recall"].mean(),
                    "roc_auc": scores["test_roc_auc"].mean(),
                    "brier_score":-scores["test_neg_brier_score"].mean()
                    })
resultdf = pd.DataFrame(results)
resultdf.sort_values("recall",ascending=False)


,n_estimators,max_depth,accuracy,precision,recall,roc_auc,brier_score
0,100,3.0,0.798797,0.800036,0.850122,0.886046,0.144716
4,200,5.0,0.810996,0.816781,0.850122,0.891061,0.133897
8,500,8.0,0.817808,0.827252,0.850122,0.892483,0.130643
9,100,NaN,0.813810,0.823643,0.847744,0.883547,0.134029
3,100,5.0,0.806960,0.813726,0.845244,0.891503,0.133835
6,100,8.0,0.808293,0.816284,0.845244,0.890368,0.131307
10,200,NaN,0.809737,0.820275,0.842866,0.883153,0.134342
11,500,NaN,0.815143,0.828924,0.842805,0.885035,0.133278
5,500,5.0,0.808293,0.817430,0.842805,0.889679,0.133768
1,200,3.0,0.796094,0.801078,0.842744,0.885747,0.144407


Best choices would be n_estimators=200, max_depth = 5 OR 4th row on the above table, n_estimators = 200, max_depth = 8 because of slightly higher precision and slightly better brier scoere, although slightly worse recall. The setup on 4th row is objectively the best, I would choose it, but it is more computational heavy than n_estimators=200, max_depth = 5 which is almost as good.

In [22]:
from xgboost import XGBClassifier

categorical = ["sex","cp","fbs","restecg","exang","slope","ca","thal"]
numerical=[]
for i in X.columns:
    if i not in categorical:
        numerical.append(i)

preprocessor_trees = make_column_transformer(
    (Pipeline([("imputer",SimpleImputer(strategy="median"))]),numerical),
(Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("encoder",OneHotEncoder(handle_unknown="ignore"))]),categorical)
)

xgbmodel = Pipeline([("preprocessor",preprocessor_trees),("classifier",XGBClassifier(n_estimators=100,learning_rate=0.05,max_depth=5,random_state=10))])
cv = StratifiedKFold(n_splits=10,shuffle=True,random_state=10)
scores = cross_validate(xgbmodel,X_train,y_train,cv=cv,scoring=["accuracy","precision","recall","roc_auc","neg_brier_score"])

for metric in ["accuracy", "precision", "recall", "roc_auc","neg_brier_score"]:
    values= scores[f"test_{metric}"]
    if metric == "neg_brier_score":
        values = -values
        metric="brier_score"
    print(f"{metric}: {values.mean():.3f} +- {values.std():.3f}")
    
    

accuracy: 0.800 +- 0.046
precision: 0.817 +- 0.055
recall: 0.828 +- 0.049
roc_auc: 0.881 +- 0.036
brier_score: 0.139 +- 0.025


In [23]:
param_grid = {"classifier__n_estimators": [100,200,500],"classifier__learning_rate":[0.02,0.03,0.05,0.07],"classifier__max_depth":[3,5,7,8]}
results=[]
for params in ParameterGrid(param_grid):
    xgbmodel.set_params(**params)
    scores = cross_validate(xgbmodel,X_train,y_train,cv=cv,scoring=["accuracy","precision","recall","roc_auc","neg_brier_score"])
    results.append({"n_estimators": params["classifier__n_estimators"],
                    "learning_rate":params["classifier__learning_rate"],
                    "max_depth": params["classifier__max_depth"],
                    "accuracy": scores["test_accuracy"].mean(),
                    "precision": scores["test_precision"].mean(),
                    "recall": scores["test_recall"].mean(),
                    "roc_auc": scores["test_roc_auc"].mean(),
                    "brier_score": -scores["test_neg_brier_score"].mean()
                    })
resultsdf = pd.DataFrame(results)
resultsdf.sort_values("recall",ascending=False)


,n_estimators,learning_rate,max_depth,accuracy,precision,recall,roc_auc,brier_score
25,200,0.05,3,0.827323,0.830535,0.867500,0.890742,0.130162
24,100,0.05,3,0.816420,0.816001,0.864939,0.890233,0.130839
13,200,0.03,3,0.823214,0.825628,0.864878,0.890395,0.130367
14,500,0.03,3,0.817771,0.819789,0.862500,0.885995,0.134144
2,500,0.02,3,0.821844,0.825155,0.862500,0.891560,0.130584
36,100,0.07,3,0.821862,0.826855,0.860000,0.891142,0.130052
1,200,0.02,3,0.808256,0.808555,0.857622,0.887633,0.133369
37,200,0.07,3,0.812310,0.816535,0.855122,0.887791,0.132997
26,500,0.05,3,0.815050,0.823419,0.850183,0.879819,0.139524
12,100,0.03,3,0.806886,0.810665,0.850122,0.884618,0.136640


Out of these the best seems to be n_estimators = 200, learning_rate = 0.05, max_depth=3 by all metrics.

It seems to me that XGBoost model edges out to be the best choice for final model. Therefore let's create the final model with XGBoost and fit on the training split and evaluate on the test split created earlier.

In [24]:


final_model = Pipeline([("preprocessor",preprocessor_trees),("classifier",XGBClassifier(n_estimators=200,learning_rate=0.05,max_depth=3,random_state=10))])
final_model.fit(X_train,y_train)
#After playing around with the threshold, lowering it a bit from default 0.5 seems to improve the model a bit.
threshold=0.40
proba = final_model.predict_proba(X_test)[:,1]
preds = (proba >= threshold).astype(int)

#metrics
accuracy = accuracy_score(y_test,preds)
confusion = confusion_matrix(y_test,preds)
TP = confusion[1,1]
FP = confusion[0,1]
FN = confusion[1,0]
precision = TP/(TP+FP)
recall = TP/(TP+FN)
auc = roc_auc_score(y_test,proba)
print(f"Confusion matrix:")
print(confusion)
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"roc-auc: {auc}")
brier = brier_score_loss(y_test,proba)
print(f"brier score: {brier}")



Confusion matrix:
[[56 26]
 [14 88]]
Accuracy: 0.782608695652174
Precision: 0.7719298245614035
Recall: 0.8627450980392157
roc-auc: 0.8586800573888091
brier score: 0.1517915490314957


Accuracy: 0.782608695652174
Precision: 0.7719298245614035
Recall: 0.8627450980392157
roc-auc: 0.8586800573888091
brier score: 0.1517915490314957

In [25]:
results = X_test.copy()
results["actual"] = y_test.values
results["predicted"] = preds
results["probability"] = proba.round(3)
results["correct"] = results["actual"] == results["predicted"]
results[["age", "sex", "cp", "actual", "predicted", "probability", "correct"]].head(10)

,age,sex,cp,actual,predicted,probability,correct
496,37,Male,asymptomatic,1,1,0.660,True
593,54,Female,non-anginal,1,1,0.427,True
445,54,Female,atypical angina,0,0,0.132,True
274,59,Male,typical angina,1,0,0.369,False
834,50,Male,asymptomatic,1,1,0.444,True
849,48,Male,asymptomatic,0,1,0.718,False
385,46,Male,asymptomatic,0,1,0.871,False
8,63,Male,asymptomatic,1,1,0.921,True
498,38,Male,asymptomatic,1,1,0.498,True
746,51,Male,asymptomatic,1,1,0.888,True


#Next cell is for app purposes, ignore

In [26]:
import joblib
final_model = Pipeline([("preprocessor",preprocessor_trees),("classifier",XGBClassifier(n_estimators=200,learning_rate=0.05,max_depth=3,random_state=10))])
final_model.fit(X, y)

joblib.dump(final_model, "heart_disease_model.joblib")

['heart_disease_model.joblib']